In [ ]:
#@title **_Installing_ dan _Importing Library_**
!pip install -U scikit-learn
!pip install Sastrawi
!pip install wordcloud

import nltk
import pickle
import re
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from wordcloud import WordCloud

import Sastrawi
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score, recall_score, f1_score
from sklearn.model_selection import KFold

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.corpus import wordnet
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

In [ ]:
#@title **_Mount Google Drive_** #Jika tidak menggunakan Google Colab, bisa dihapus/abaikan
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#@title **Membaca data ulasan yang telah melalui pra-pemrosesan teks**
df = pd.read_csv('labeled.csv')

In [ ]:
df['label'].value_counts()

In [ ]:
#@title Menghapus baris bersentimen netral
df = df[df['label'] != 'netral']

df.head(10)

In [ ]:
len(df)

In [ ]:
# @title **Data 1.1 - Split dataset menjadi 80% data latih, and 20% data uji**
data_train, data_test, y_train, y_test = train_test_split(df['content'].values, df_1['label'].values
, test_size=0.2, random_state=42)

In [ ]:
# @title **TF-IDF Vectorizer**
vectorizer = TfidfVectorizer()
xtrain = vectorizer.fit_transform(data_train)
xtest = vectorizer.transform(data_test)

In [ ]:
# @title **Pelatihan Model**
model = SVC(kernel='linear', random_state=42)
model.fit(xtrain, y_train)

In [ ]:
# @title **Prediksi**
predict = model.predict(xtest)
accuracy = accuracy_score(y_test, predict)
print(f"Akurasi: {accuracy}")
print (classification_report(y_test, predict))

In [ ]:
# @title **Confusion Matrix**
cm_1 = confusion_matrix(y_test, predict)
print(cm_1)

In [ ]:
# @title *Evaluasi Model dengan K-Fold Cross Validation*
kf = KFold(n_splits=5, shuffle=True, random_state=42)
accuracies = []
precisions = []
recalls = []
f1_scores = []
fold_num = 1
confusion_matrices = []

all_predictions = []

for train_index, test_index in kf.split(df['content']):
    print(f"Fold Data - {fold_num}:")
    fold_num += 1
    data_trains, data_test = df['content'].iloc[train_index], df['content'].iloc[test_index]
    y_trains, y_tests = df['label'].iloc[train_index], df['label'].iloc[test_index]

    # TF-IDF
    vectorizers = TfidfVectorizer()
    xtrains = vectorizers.fit_transform(data_trains)
    xtests = vectorizers.transform(data_tests)

    # Melatih Model
    models = SVC(kernel='linear', random_state=42)
    models.fit(xtrains, y_trains)

    # Evaluasi Model
    predicts = models.predict(xtests)
    accuracys = accuracy_score(y_tests, predicts)
    cms = confusion_matrix(y_tests, predicts)
    precisions.append(precision_score(y_tests, predicts, average='weighted'))
    recalls.append(recall_score(y_tests, predicts, average='weighted'))
    f1_scores.append(f1_score(y_tests, predicts, average='weighted'))
    accuracies.append(accuracys)
    confusion_matrices.append(cms)

    average_cm = np.mean(confusion_matrices, axis=0)

    print(classification_report(y_tests, predicts))
    print(f"Akurasi Fold {fold_num - 1}: {accuracys}\n")


# Rata-rata akurasi di semua fold
print("Rata-rata akurasi :", np.mean(accuracies))
print("Rata-rata presisi :", np.mean(precisions))
print("Rata-rata recall :", np.mean(recalls))
print("Rata-rata F1-score :", np.mean(f1_scores))

In [ ]:
#@title **Data 1.1 - Evaluasi Model (_Confusion Matrix_)**
labels = ['Negatif', 'Positif']
sns.heatmap(average_cm, annot=True,cmap='Blues',
            fmt='.2f', xticklabels=labels, yticklabels=labels)
plt.xlabel('Predicted')
plt.gca().xaxis.set_label_position('top')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.savefig('confusion_matrix_1.png')
plt.show()

In [ ]:
print(average_cm)